# 2c_HM_number_distribution_stats

Count-answer distribution statistics for the matched HM comparison set across all three control variants. Plot generation lives in `figures/answer_distribution/answer_distribution.py`; this notebook is statistical-only.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

_NOTEBOOK_DIR = Path.cwd()
if (_NOTEBOOK_DIR / "helpers.py").exists():
    sys.path.insert(0, str(_NOTEBOOK_DIR.parent))
elif (_NOTEBOOK_DIR / "notebooks" / "helpers.py").exists():
    sys.path.insert(0, str(_NOTEBOOK_DIR))

from notebooks.helpers import (
    ROOT,
    LATEX_TABLES,
    pretty_print_path,
    hh_question_means,
    hm_question_means,
    variant_summary_table,
    pairwise_variant_correlation_table,
    grouped_pattern_table,
    scatter_correlation_table,
    blind_accuracy_summary,
    qualitative_qdf,
    attach_answer_summaries,
    hh_ranked_examples,
    variant_top_bottom_table,
    hh_degradation_table,
    top_questions_tables,
    to_latex_table,
)

from notebooks.helpers import number_distribution_table


In [ ]:
VARIANT_LABELS = {"C": "Original", "B": "Weaker", "A": "Pronominalized"}

human_number_profiles = {}
number_inst_tables = []
for variant in ["C", "B", "A"]:
    human_profile, table = number_distribution_table(condition="inst_blind", variant=variant)
    human_number_profiles[variant] = human_profile
    number_inst_tables.append(table.assign(Variant=VARIANT_LABELS[variant]))

number_inst = pd.concat(number_inst_tables, ignore_index=True)
for variant in ["C", "B", "A"]:
    print(VARIANT_LABELS[variant])
    display(human_number_profiles[variant])
    display(number_inst[number_inst["Variant"] == VARIANT_LABELS[variant]])

In [ ]:
out = LATEX_TABLES / "hm_number_distribution_inst_blind.tex"
to_latex_table(
    number_inst,
    out,
    "Count-answer distribution statistics against the human reference across control variants (instruction-aware blind condition).",
    "tab:hm_number_distribution_inst_blind",
    float_formatters={"JS divergence": ".3f", "TV distance": ".3f", "Chi-square": ".2f", "p": ".1e", "0": ".3f", "1": ".3f", "2–3": ".3f", "4–5": ".3f", "6–10": ".3f", "11–20": ".3f", ">20": ".3f", "others": ".3f"},
)
print(pretty_print_path(out))

In [ ]:
number_blind_tables = []
for variant in ["C", "B", "A"]:
    _, table = number_distribution_table(condition="blind", variant=variant)
    number_blind_tables.append(table.assign(Variant=VARIANT_LABELS[variant]))

number_blind = pd.concat(number_blind_tables, ignore_index=True)
display(number_blind)

In [ ]:
ranked_number = number_inst.sort_values(["Variant", "JS divergence", "TV distance", "Model"]).reset_index(drop=True)
for variant in ["Original", "Weaker", "Pronominalized"]:
    sub = ranked_number[ranked_number["Variant"] == variant]
    best_num = sub.iloc[0][["Variant", "Model", "Group", "JS divergence", "TV distance", "Chi-square", "p", "Significant"]]
    worst_num = sub.iloc[-1][["Variant", "Model", "Group", "JS divergence", "TV distance", "Chi-square", "p", "Significant"]]
    print(f"Closest count distribution to humans ({variant}, inst_blind):")
    display(best_num.to_frame().T)
    print(f"Farthest count distribution from humans ({variant}, inst_blind):")
    display(worst_num.to_frame().T)

print("Interpretation: models are ranked by closeness to the human count distribution using JS divergence and TV distance, with chi-square reported as the significance test on aggregate category counts.")